# RAG 기반 Customer Service AI 에이전트 개발

---

## 학습 개요

### 학습 주제
- **LLM의 한계와 RAG의 필요성**: 환각(Hallucination) 현상, 도메인 지식 부재, 최신 정보 미반영 등 LLM 단독 사용의 한계를 이해하고 RAG로 해결하는 방법
- **검색 기반 문서 탐색**: 키워드 검색(Lexical Search)과 의미 기반 검색(Semantic Search)의 원리 및 장단점 비교
- **RAG 파이프라인 구축**: 문서 로딩 → 청킹 → 임베딩 → Vector Store 저장 → 검색 → LLM 답변 생성까지의 전체 흐름

### 학습 목표
- LLM의 환각(Hallucination) 현상을 직접 확인하고, RAG가 필요한 상황을 설명할 수 있다
- 키워드 검색과 의미 기반 검색의 차이점을 코드로 구현하고 결과를 비교할 수 있다
- 텍스트 임베딩의 원리(코사인 유사도)를 이해하고 Vector Store에 문서를 저장할 수 있다
- 청킹 전략(chunk_size, overlap)에 따른 검색 품질 차이를 분석하고 적절한 설정을 선택할 수 있다
- LangGraph StateGraph를 사용하여 검색-생성 RAG 파이프라인을 구현하고 실행할 수 있다

### 핵심 개념
- **RAG(Retrieval-Augmented Generation)**: LLM의 환각/지식 한계를 외부 문서 검색으로 보완하는 기법. 질문 → 관련 문서 검색 → 검색 결과를 컨텍스트로 포함하여 LLM에 전달 → 정확한 답변 생성의 흐름으로 동작
- **임베딩(Embedding)**: 텍스트를 고차원 벡터 공간(예: 1536차원)에 표현하는 기술. 의미가 유사한 텍스트는 벡터 공간에서 가까운 위치에 배치되어 코사인 유사도로 유사성 측정 가능
- **Vector Store**: 임베딩 벡터를 저장하고 유사도 기반 검색(Approximate Nearest Neighbor)을 수행하는 특수 데이터베이스. ChromaDB, FAISS, Pinecone 등이 대표적
- **청킹(Chunking)**: 긴 문서를 검색에 적합한 크기의 조각(chunk)으로 분할하는 기법. `chunk_size`(청크 최대 길이)와 `chunk_overlap`(청크 간 겹침)을 조절하여 검색 정확도와 문맥 유지의 균형 조정
- **Retriever**: 사용자 쿼리를 임베딩하고 Vector Store에서 유사도가 높은 문서(top-k)를 검색하여 반환하는 컴포넌트. LangChain에서 `vectorstore.as_retriever()`로 생성

### 선행 지식
- **Python 프로그래밍**: 클래스, 함수, 딕셔너리, 리스트 컴프리헨션, 라이브러리 import/사용 (중급)
- **LLM API 사용 경험**: OpenAI API 호출, 프롬프트 작성, 응답 파싱 경험
- **기본 수학 개념**: 벡터, 유사도(거리) 개념에 대한 기초 이해 (코사인 유사도 공식은 실습에서 설명)
- **선수 학습**: Chapter 3 LLM 기초 실습 완료 권장

---

## 실습 구성

### 학습 방향

- **실습 구성 방식**
  - 문제와 정답 코드가 병렬로 제공되며, 각 단계별로 TODO 영역을 채우며 학습자가 직접 코드를 작성합니다.

- **Required Package**
  - `langchain>=0.3.0`
  - `langchain-openai>=0.2.0`
  - `langchain-upstage>=0.2.0`
  - `langchain-community>=0.3.0`
  - `langgraph>=0.2.0`
  - `chromadb>=0.4.0`
  - `tiktoken>=0.5.0`
  - `python-dotenv>=1.0.0`
  - `pymupdf>=1.24.0`

- **Step 요약**
  - **Step 1 (5분)**: 환경 설정 — 라이브러리 설치 및 API Key 설정
  - **Step 2 (5분)**: LLM의 한계 체감 — "LLM 혼자서는 정확한 답변이 어렵다"
  - **Step 3 (10분)**: 자료와 함께하면 해결 — "자료가 있으면 정확해진다 → 어떻게 찾을까?"
  - **Step 4 (10분)**: 키워드로 자료 찾기 — "키워드 일치만 찾는다 → 의미도 찾으려면?"
  - **Step 5 (10분)**: 의미 기반 검색 + DB화 기초 — "단위가 크면 부정확하다 → 더 잘게 나눠야 하나?"
  - **Step 6 (10분)**: 청킹 전략 — 상황별 청킹 전략 선택 기준
  - **Step 7 (10분)**: 전체 RAG 파이프라인 (LangGraph) — Step 2와 비교하여 RAG 효과 확인

### 문제 설명

- **문제 개요**: 이 실습은 **"LLM만으로는 왜 안 되는가?"**라는 질문에서 시작합니다. 학습자는 LLM의 한계(환각, 도메인 지식 부재)를 직접 확인하고, **키워드 검색 → 의미 기반 검색 → 청킹 전략**으로 단계별 해결책을 발견하며, 최종적으로 **LangGraph StateGraph를 활용한 RAG 파이프라인**으로 정확한 고객 서비스 답변을 생성할 수 있어야 합니다.
- **요구사항 요약**
  - **LLM 한계 확인**: 환각 현상과 도메인 지식 부재를 직접 체감
  - **검색 방식 비교**: 키워드 검색 vs 의미 기반 검색의 차이점 이해
  - **Vector Store 활용**: 문서 임베딩 및 ChromaDB 저장/검색
  - **청킹 전략 적용**: chunk_size, overlap에 따른 검색 품질 차이 분석
  - **RAG 파이프라인 구현**: LangGraph StateGraph로 retrieve → generate 워크플로우 구현

### 데이터셋 개요 및 저작권 정보

- **데이터셋 명**: Yes24 서비스 혜택 문서
- **데이터셋 개요**: 온라인 서점 Yes24의 배송 정책, 포인트 제도, 회원 혜택 등을 담은 PDF 문서입니다.
- **사용 목적**: RAG 파이프라인 구축을 위한 검색 대상 문서
- **저작권/출처**: Yes24 공개 서비스 안내 (교육 목적 사용)
- **주의사항**: API Key 환경 변수 설정이 필요합니다. 실제 서비스 정책은 변경될 수 있으므로 학습 목적으로만 사용하세요.

### 학습 문제

- **Step 2 — LLM의 한계 체감**
  - **TODO 1**: LLM에게 직접 질문하기 *(연결: 환각/Hallucination, 도메인 지식 부재)*
  - **1줄 요약**: LLM 단독 사용 시 환각 현상을 직접 확인한다.
- **Step 3 — 자료와 함께하면 해결**
  - **TODO 2**: 자료를 직접 프롬프트에 넣어 질문하기 *(연결: 컨텍스트 기반 답변)*
  - **1줄 요약**: 관련 자료를 프롬프트에 포함하면 정확도가 향상됨을 확인한다.
- **Step 4 — 키워드로 자료 찾기**
  - **TODO 3**: 키워드로 관련 문서 검색하기 *(연결: Lexical Search의 한계)*
  - **1줄 요약**: 키워드 검색은 정확히 일치하는 단어만 찾을 수 있음을 체감한다.
- **Step 5 — 의미 기반 검색 + DB화 기초**
  - **TODO 4**: 문서를 Vector DB에 저장하기 *(연결: 임베딩/Embedding, Vector Store)*
  - **TODO 5**: 의미 기반 검색 수행하기 *(연결: Semantic Search, 코사인 유사도)*
  - **1줄 요약**: 임베딩을 통해 의미가 유사한 문서를 검색할 수 있다.
- **Step 6 — 청킹 전략**
  - **TODO 6**: 청킹 전략 비교하기 *(연결: Chunking, chunk_size/overlap 트레이드오프)*
  - **1줄 요약**: 청킹 설정에 따라 검색 정확도와 문맥 유지의 균형이 달라진다.
- **Step 7 — 전체 RAG 파이프라인 (LangGraph)**
  - **TODO 7**: LangGraph StateGraph로 RAG 파이프라인 구현 *(연결: RAG, LangGraph StateGraph, Retriever)*
  - **1줄 요약**: 검색-생성 워크플로우를 LangGraph로 구현하고 RAG 효과를 확인한다.

---
# Step 1: 환경 설정

### Setup: 라이브러리 설치 및 API Key 설정

In [2]:
# 패키지 설치
# %%capture
%pip install -y "langchain>=0.3.0" "langchain-openai>=0.2.0" "langchain-upstage>=0.3.0" "langchain-community>=0.3.0" "langgraph>=0.2.0" "langchain-text-splitters>=0.3.0" "chromadb>=0.5.0" "tiktoken>=0.7.0" "pymupdf>=1.24.0" "python-dotenv>=1.0.0"

Note: you may need to restart the kernel to use updated packages.



Usage:   
  c:\Users\SSAFY\AppData\Local\Programs\Python\Python312\python.exe -m pip install [options] <requirement specifier> [package-index-options] ...
  c:\Users\SSAFY\AppData\Local\Programs\Python\Python312\python.exe -m pip install [options] -r <requirements file> [package-index-options] ...
  c:\Users\SSAFY\AppData\Local\Programs\Python\Python312\python.exe -m pip install [options] [-e] <vcs project url> ...
  c:\Users\SSAFY\AppData\Local\Programs\Python\Python312\python.exe -m pip install [options] [-e] <local project path> ...
  c:\Users\SSAFY\AppData\Local\Programs\Python\Python312\python.exe -m pip install [options] <archive url/path> ...

no such option: -y


In [3]:
import os
import warnings
from dotenv import load_dotenv
warnings.filterwarnings("ignore")

# .env 파일에서 환경변수 로드
load_dotenv()

# API 키 확인 (.env 미설정 시 직접 입력)
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = input("🔑 OPENAI_API_KEY를 입력하세요: ")

if not os.environ.get("UPSTAGE_API_KEY"):
    os.environ["UPSTAGE_API_KEY"] = input("🔑 UPSTAGE_API_KEY를 입력하세요: ")
print("✅ 환경 설정 완료")

✅ 환경 설정 완료


In [4]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

# ═══════════════════════════════════════════════════════════════
# 모델 초기화 — OpenAI / Solar 중 택 1 (주석 해제하여 전환)
# ═══════════════════════════════════════════════════════════════

# ── Option A: OpenAI (기본) ──
# llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
# embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# ── Option B: Solar 대안 (주석 해제하여 사용) ──
from langchain_upstage import ChatUpstage, UpstageEmbeddings
llm = ChatUpstage(model="solar-pro3")
embeddings = UpstageEmbeddings(model="embedding-query")

In [5]:
# 모델 연결 테스트
response = llm.invoke("Yes24 배송 정책에 대해 알려줘.")
print("LLM 응답:")
print(response.content[:200] + "...")
print(f"\n✅ 모델 연결 확인 완료 ({llm.model_name})")

AuthenticationError: Error code: 401 - {'error': {'message': 'Your API key is invalid. Please verify your API key or generate a new one from our API keys management page (https://console.upstage.ai/api-keys). Remember to keep your API keys secure and never share them publicly', 'type': 'invalid_request_error', 'param': '', 'code': 'invalid_api_key'}}

In [ ]:
# 데이터 파일 경로 설정
# 실습 폴더 내 data/ 디렉토리에 PDF 파일이 위치해야 합니다.
# 예시 디렉토리 구조:
#   Practice/
#   ├── (실습-답안) 4-1_RAG_기반_Customer_Service_AI_에이전트_개발.ipynb  ← 현재 파일
#   └── data/
#       ├── 총알배송_서비스 혜택 - 예스24.pdf
#       ├── 배송지연 보상제도_서비스 혜택 - 예스24.pdf
#       └── ...
DATA_DIR = "data/"

# 데이터 파일 목록 확인
import glob
import os

pdf_files = glob.glob(DATA_DIR + "*.pdf")
print(f"📁 발견된 PDF 파일: {len(pdf_files)}개")
for f in pdf_files:
    print(f"  - {os.path.basename(f)}")

if len(pdf_files) == 0:
    print(f"\n⚠️ PDF 파일이 발견되지 않았습니다.")
    print(f"   '{DATA_DIR}' 경로에 PDF 파일을 배치해주세요.")

📁 발견된 PDF 파일: 8개
  - 도서 품절 보상제도 - 예스24.pdf
  - 매장 픽업 서비스_서비스 혜택 - 예스24.pdf
  - 무료배송 추가적립_서비스 혜택 - 예스24.pdf
  - 배송지연 보상제도_서비스 혜택 - 예스24.pdf
  - 신규 회원 혜택_서비스 혜택 - 예스24.pdf
  - 영원한 YES포인트_서비스 혜택 - 예스24.pdf
  - 제휴 할인카드_서비스 혜택 - 예스24.pdf
  - 총알배송_서비스 혜택 - 예스24.pdf


---
# Step 2: LLM의 한계 체감

### Concept Check: LLM의 한계

LLM(Large Language Model)은 방대한 텍스트 데이터로 학습되어 다양한 질문에 답변할 수 있습니다. 
하지만 다음과 같은 **근본적인 한계**가 있습니다:

1. **환각(Hallucination)**: 학습 데이터에 없는 내용을 그럴듯하게 지어내는 현상
2. **최신 정보 부재**: 학습 데이터 이후의 정보는 알 수 없음
3. **도메인 지식 부족**: 특정 회사/서비스의 내부 정보는 학습되지 않음

아래 TODO에서 LLM에게 **Yes24 배송 정책**에 대해 질문하고, 답변이 정확한지 확인해봅니다.

### TODO 1: LLM에게 직접 질문하기

- **요구사항**: OpenAI GPT 모델에게 Yes24 배송 정책에 대해 질문하고 답변 확인
- **입력**: 고객 서비스 질문 (예: "Yes24에서 총알배송이 뭔가요?")
- **출력**: LLM 응답
- **확인**: 답변이 정확한지, 환각이 있는지 판단
- **힌트**: `ChatOpenAI`, `HumanMessage` 사용

**예상 결과:**
- `response` 객체에 LLM의 응답이 담겨 있어야 함
- `response.content`로 응답 텍스트 확인 가능
- LLM이 Yes24의 정확한 정책을 모르므로 일반적이거나 부정확한 답변이 나올 것

**정상 동작 확인 방법:**
- 출력에 "🤖 LLM 응답:" 다음에 텍스트가 출력되면 성공
- "⚠️ 코드를 완성해주세요!" 메시지가 나오면 코드 수정 필요

In [ ]:
# TODO: LLM에게 직접 질문하기
from langchain_core.messages import HumanMessage

# Yes24 서비스에 대한 질문
question = "Yes24에서 총알배송이 뭔가요? 어떤 조건에서 가능한가요?"

# LLM에게 직접 질문
response = llm.invoke([HumanMessage(content=question)])

print("📝 질문:", question)
print("\n🤖 LLM 응답:")
print(response.content)

📝 질문: Yes24에서 총알배송이 뭔가요? 어떤 조건에서 가능한가요?

🤖 LLM 응답:
**Yes24 총알배송(프라임 배송, 새벽·당일·익일 배송) 요약**

| 구분 | 주요 특징 | 적용 조건 | 배송 기간 | 이용 방법 |
|------|-----------|-----------|----------|----------|
| **당일 배송** | 주문 후 1 ~ 2 시간 안에 배송 완료 | - 주문 시간이 **오전 10시 ~ 오후 12시** 사이 <br> - 재고가 **당일 출고 가능한 상품** (도서·음반·DVD 등 일부) <br> - 배송지가 **서울·수도권** (일부 지방도 가능) | 주문 시점 → 1~2 시간 내 | - ‘당일 배송’ 옵션을 선택하거나, 상품이 ‘당일 배송 가능’으로 표시될 때 자동 적용 |
| **새벽 배송** | 주문 당일 저녁 9시 ~ 새벽 2시 사이에 배송 | - 주문 시간이 **오후 4시 ~ 저녁 8시** <br> - 재고가 **당일 출고 가능한 상품** <br> - 배송지가 **서울·수도권** (일부 지방도 가능) | 주문 시점 → 당일 저녁~새벽 | - ‘새벽 배송’ 옵션을 체크하거나, ‘프라임’ 회원이면 자동 제공 |
| **익일 배송** (프라임) | 주문 다음 날 도착 (보통 9시 이전) | - **프라임 멤버십 가입** (월/연회비) <br> - 주문 시간이 **당일 오후 12시 이전까지** <br> - 재고가 **프라임 전용 출고 상품** (대부분 도서·음반·DVD) | 주문 → 다음 날 9시 이전 | - 프라임 회원이면 기본 옵션, 일반 주문은 ‘익일 배송’ 선택 시 추가 비용 발생 |
| **총알배송** (프라임 + 특별 할인) | ‘총알배송’ 라벨이 붙은 상품만 적용, 프라임 회원이라면 **추가 비용 없이** 익일 배송 | - **프라임 멤버십** <br> - ‘총알배송’ 표시된 도서·음반·DVD 등 <br> - 주문 시간: 프라임 기준(보통 **당일 오후 12시** 전까지) <br> - 지역 제한: 수

In [ ]:
# 다른 질문도 테스트
test_questions = [
    "Yes24 배송지연 보상제도는 어떻게 되나요?",
    "Yes24에서 YES포인트 적립률은 얼마인가요?",
    "Yes24 신규 회원 혜택은 무엇인가요?"
]

for q in test_questions:
    response = llm.invoke([HumanMessage(content=q)])
    print(f"\n📝 질문: {q}")
    print(f"🤖 응답: {response.content[:1000]}...")
    print("-" * 50)


📝 질문: Yes24 배송지연 보상제도는 어떻게 되나요?
🤖 응답: ## Yes24(Yes24.com) 배송 지연 보상 제도 개요  

아래는 2025년 7월 까지의 정책·약관 정보를 바탕으로 정리한 내용이며, 구체적인 보상 조건·절차·금액은 **각 판매자·배송 파트너·Yes24 자체 정책**에 따라 달라질 수 있습니다.  
구매 전·후에 반드시 해당 상품 페이지·주문 상세·또는 고객센터에서 최신 정보를 확인하시기 바랍니다.

---

### 1. 기본 전제  

| 구분 | 내용 |
|------|------|
| **보상 적용 가능 시점** | 주문 후 **예정 배송일(delivery date)**이 지나고도 상품이 도착하지 않을 때. 예정 배송일은 주문 상세 페이지·배송 알림·이메일 등에 표시됩니다. |
| **보상 대상** | 구매한 상품(디지털·실물 모두) 중 **배송이 지연된 경우**. 반품·취소·해외 직수입·주문 취소 등 다른 사유로 인한 지연은 별도 정책이 적용됩니다. |
| **보상 제외 사유** | - 자연재해·천재지변, 국가·지역 차원의 물류 파업·통관 지연 등 **불가항력** 상황 <br> - 구매자가 배송 주소를 잘못 입력하거나, 배송지를 변경한 경우 <br> - 판매자가 직접 배송을 담당하지 않고 **제3자 물류**(예: 직접 배송, 해외 직배송)로 인한 지연 <br> - 주문 후 **상품 자체가 품절·취소**된 경우(보상보다는 주문 취소·환불 절차) |

> **핵심 포인트**  
> - Yes24는 **‘배송 지연 보상’**을 별도로 규정하고 있지는 않지만, **판매자와 연계된 내부 보상 프로그램**(예: 포인트 적립·쿠폰·할인)으로 보상을 제공하는 경우가 있습니다.  
> - 따라서 실제 보상은 **‘판매자 자체 보상 정책’**에 따라 진행됩니다.  

---

### 2. 보상 형태 (대표적인 경우)

| 보상 형태 | 설명 | 적용 시 예시 |
|-----------|------|------------|
| **포인트 적립*

### Test: LLM 한계 확인

위 응답을 보면:
- LLM이 Yes24의 **구체적인 정책 내용을 정확히 알지 못함**
- 일반적인 추측이나 **환각(Hallucination)**으로 답변할 가능성이 있음
- 실제 서비스 정책과 다를 수 있어 **고객 서비스에 직접 사용하기 어려움**

**정상 동작 기준:**
- LLM이 응답을 생성하되, Yes24의 정확한 정책(총알배송 조건, 배송 시간 등)을 알지 못함
- "일반적으로", "아마도", "확인이 필요합니다" 등의 불확실한 표현이 포함될 수 있음

**예상 출력 예시:**
```
📝 질문: Yes24에서 총알배송이 뭔가요? 어떤 조건에서 가능한가요?

🤖 LLM 응답:
총알배송은 Yes24에서 제공하는 빠른 배송 서비스로 추정됩니다. 
일반적으로 이런 서비스는 특정 지역이나 상품에 한해 제공되며...
(구체적인 조건이나 시간은 정확하지 않을 수 있음)
```

**핵심 포인트:** LLM 혼자서는 도메인 특화 정보에 대해 정확한 답변을 제공하기 어렵습니다.

---
# Step 3: 자료와 함께하면 해결

### Concept Check: 컨텍스트 추가하기

LLM의 한계를 극복하는 가장 간단한 방법은 **관련 자료를 프롬프트에 직접 포함**시키는 것입니다.

```
# 기존 방식 (환각 위험)
"Yes24 총알배송이 뭔가요?"

# 개선된 방식 (컨텍스트 포함)
"다음 자료를 참고해서 답변해주세요:
[총알배송 정책 문서 내용...]

질문: Yes24 총알배송이 뭔가요?"
```

이 방식의 장점:
- LLM이 **실제 자료를 기반**으로 답변
- **환각 가능성 대폭 감소**
- **최신 정보** 반영 가능

### TODO 2: 자료를 직접 프롬프트에 넣어 질문하기

- **요구사항**: PDF 문서에서 텍스트를 추출하고, 이를 프롬프트에 포함시켜 LLM에게 질문
- **입력**: PDF 문서 내용 + 질문
- **출력**: 자료 기반 LLM 응답
- **확인**: Step 2 결과와 비교하여 정확도 개선 확인
- **힌트**: `PyMuPDFLoader`, `ChatPromptTemplate`, `llm.invoke()` 사용

**예상 결과:**
- `prompt_template`이 `ChatPromptTemplate` 객체여야 함
- 프롬프트에 `{context}`와 `{question}` 플레이스홀더가 포함되어 있어야 함
- 응답이 문서 내용을 기반으로 구체적인 정책 정보를 포함해야 함

**정상 동작 확인 방법:**
- "🤖 자료 기반 응답:" 다음에 총알배송의 구체적인 조건(예: 배송 시간, 가능 지역 등)이 출력되면 성공
- Step 2의 응답보다 구체적이고 정확한 정보가 포함되어 있으면 정상

In [ ]:
# TODO 2: 자료를 직접 프롬프트에 넣어 질문하기
from langchain_community.document_loaders import PyMuPDFLoader

# 총알배송 문서 로드
pdf_path = DATA_DIR + "총알배송_서비스 혜택 - 예스24.pdf"
loader = PyMuPDFLoader(pdf_path)
documents = loader.load()

# 문서 내용 확인
doc_content = "\n".join([doc.page_content for doc in documents])
print(f"📄 문서 길이: {len(doc_content)} 글자")
print(f"\n📄 문서 내용 (일부):\n{doc_content[:500]}...")

📄 문서 길이: 2508 글자

📄 문서 내용 (일부):
싸다
빠르다
믿을 수 있다
전국 방방곡곡으로 총알배송 서비스는 계속 확대됩니다.
당일배송 표시상품을 주문하시면 당일에 받으실 수 있습니다.
주문일
원주
춘천
서울
수도권
(평택제외)
인천
대전, 세종
청주, 천안
아산, 평택
광주
울산, 창원
포항
부산, 김해
대구
경산
구미
도착
예상일
월~금
0~11시
0~13시
0~13시
0~12시
0~12시
0~12시
0~12시
0~13시
0~13시
오늘
토요일
0~11시
0~12시
0~12시
0~11시
0~12시
0~11시
0~12시
0~12시
0~12시
오늘
사무실로 주문하시는 경우 퇴근 등으로 당일 배송이 안될 수 있습니다.
오후 늦게라도 상품 수령이 가능한 주소를 입력해 주시면 더 원활한 당일배송이 가능합니다.
당일배송은 공휴일(일요일 제외)에도 일부 지역에 배송이 가능합니다.
하루배송 표시상품을 주문하시면 다음날에 받으실 수 있습니다.
지역
시간
서울
수도권(평택제외), 인천
대전, 세종, 청주, 천안,
아산, 평택
광주
대구
경산,...


In [ ]:
# 컨텍스트를 포함한 프롬프트 구성
from langchain_core.prompts import ChatPromptTemplate

prompt_template = ChatPromptTemplate.from_messages([
    ("system", """당신은 AI 온라인 서점의 고객 서비스 상담원입니다.
다음 자료를 참고하여 고객의 질문에 정확하게 답변해주세요.
자료에 없는 내용은 "해당 정보는 제공된 자료에 없습니다"라고 답변하세요.

[참고 자료]
{context}"""),
    ("human", "{question}")
])

# 질문
question = "Yes24에서 총알배송이 뭔가요? 어떤 조건에서 가능한가요?"

# 프롬프트 생성 후 LLM 직접 호출
messages = prompt_template.format_messages(context=doc_content, question=question)
response = llm.invoke(messages)

print("📝 질문:", question)
print("\n🤖 자료 기반 응답:")
print(response.content)

📝 질문: Yes24에서 총알배송이 뭔가요? 어떤 조건에서 가능한가요?

🤖 자료 기반 응답:
**총알배송**은 주문한 상품을 **최대한 빠르게 배송**해드리는 서비스입니다.  
일반 배송보다 신속하게 상품을 받을 수 있으며, 다음과 같은 조건에서 이용 가능합니다.

---

### ✅ 총알배송 이용 조건

| 조건 | 내용 |
|------|------|
| **도착 지역** | - 서울<br>- 수도권(평택 제외), 인천<br>- 대전, 세종, 청주, 천안<br>- 아산, 평택<br>- 광주<br>- 울산, 창원, 포항<br>- 부산, 김해<br>- 대구<br>- 경산, 구미<br>- 원주, 춘천 |
| **도착 시간** | - **0~11시**<br>- **0~12시**<br>- **0~13시** (지역에 따라 다름) |
| **주문 마감 시간** | - **서울, 수도권**: 평일 당일배송 종료 직후 ~ 22시까지<br>- **토요일 21시**<br>- **그 외 지역**: 평일 당일배송 종료 직후 ~ 18시까지<br>- **일요일 0시 ~ 15시** |
| **주문일** | - **평일(월~금)**: 당일 또는 다음날 배송<br>- **토요일**: 월요일 또는 화요일 배송<br>- **일요일**: 월요일 아침 배송 |
| **상품 선택** | - **당일배송/하루배송** 표시가 있는 상품에 한함<br>- 일부 상품은 제외될 수 있음 (상품 상세 페이지의 배송 안내 확인 필요) |
| **배송 불가 조건** | - 분철 옵션을 선택한 경우<br>- 사무실 등 출입 제한이 있는 장소 (직접 수령이 어려울 수 있음)<br>- 공휴일(일요일 제외), 일부 지역에 따라 배송이 제한될 수 있음 |

---

### 🔔 유의사항

- **전국 방방곡곡으로 총알배송 서비스는 계속 확대**되고 있습니다.
- 물류센터의 사정에 따라 **배송 시간이 일부 조정될 수** 있습니다.
- **원주, 춘천, 기타 지역**은 배송 시간이 다를 수 있으며, **도착 예상일은 주문 

### Test: Step 2 vs Step 3 비교

| 비교 항목 | Step 2 (LLM만) | Step 3 (자료 포함) |
|----------|----------------|--------------------|
| 정확도 | 낮음 (추측/환각) | 높음 (자료 기반) |
| 신뢰도 | 낮음 | 높음 |
| 한계 | 도메인 지식 부재 | 자료 크기 제한 |

**정상 동작 기준:**
- Step 3의 응답이 Step 2보다 구체적이고 정확해야 함
- 문서에 있는 실제 정책 내용(배송 시간, 조건 등)이 응답에 포함되어야 함

**예상 출력 예시:**
```
📝 질문: Yes24에서 총알배송이 뭔가요? 어떤 조건에서 가능한가요?

🤖 자료 기반 응답:
총알배송은 Yes24에서 제공하는 당일/익일 배송 서비스입니다.
- 오후 2시 이전 주문 시 당일 출고
- 수도권 지역 대상
- 총알배송 표시 상품에 한함
(문서에 있는 실제 정책 정보 기반)
```

**핵심 포인트:** 관련 자료를 제공하면 LLM이 정확한 답변을 생성할 수 있습니다. 그렇다면 관련 자료를 어떻게 찾을 수 있을까요?

---
# Step 4: 키워드로 자료 찾기

### Concept Check: 문자열 검색 방식

여러 문서가 있을 때, 질문과 관련된 문서를 찾는 가장 간단한 방법은 **키워드 검색**입니다.

```python
# 키워드 검색 예시
if "총알배송" in document:
    return document
```

**키워드 검색의 특징:**
- ✅ 장점: 단순하고 빠름
- ❌ 단점: **정확히 일치하는 키워드만** 검색 가능
  - "빠른 배송" → "총알배송" 문서를 못 찾음
  - "배송 빨리" → "총알배송" 문서를 못 찾음

### TODO 3: 키워드로 관련 문서 검색하기

- **요구사항**: 여러 PDF 문서에서 키워드를 포함한 문서 찾기
- **입력**: 검색 키워드 (예: "배송", "포인트")
- **출력**: 키워드를 포함한 문서 목록
- **확인**: 키워드 검색의 한계 체감 (동의어, 유사 표현 못 찾음)

**예상 결과:**
- `keyword_search` 함수가 키워드가 포함된 문서 리스트를 반환해야 함
- "총알배송" 검색 시 1개 이상의 문서가 발견되어야 함
- "빠른 배송", "신속 배송" 등 유사 표현 검색 시 0개 문서가 발견되어야 함 (한계 체감)

**정상 동작 확인 방법:**
- "🔍 키워드 '총알배송' 검색 결과: N개 문서" (N >= 1)가 출력되면 성공
- 키워드 검색 테스트에서 "총알배송"만 결과가 나오고 나머지는 0개면 정상

In [ ]:
# TODO: 키워드로 관련 문서 검색하기

# 모든 PDF 문서 로드
all_documents = []
for pdf_path in pdf_files:
    loader = PyMuPDFLoader(pdf_path)
    docs = loader.load()
    # 파일명을 메타데이터에 추가
    for doc in docs:
        doc.metadata["source_file"] = os.path.basename(pdf_path)
    all_documents.extend(docs)

print(f"📚 총 {len(all_documents)}개 페이지 로드 완료")

📚 총 8개 페이지 로드 완료


In [ ]:
# 키워드 검색 함수
def keyword_search(documents, keyword):
    """문서 리스트에서 키워드를 포함한 문서 검색"""
    results = []
    for doc in documents:
        if keyword in doc.page_content:
            results.append(doc)
    return results

# 키워드 검색 테스트
keyword = "총알배송"
results = keyword_search(all_documents, keyword)
print(f"🔍 키워드 '{keyword}' 검색 결과: {len(results)}개 문서")
for doc in results[:2]:
    print(f"  - {doc.metadata.get('source_file', 'Unknown')}")

🔍 키워드 '총알배송' 검색 결과: 8개 문서
  - 도서 품절 보상제도 - 예스24.pdf
  - 매장 픽업 서비스_서비스 혜택 - 예스24.pdf


In [ ]:
# 키워드 검색의 한계 테스트
test_keywords = [
    "빠른 배송",      # 총알배송과 유사한 의미
    "배송 빨리",      # 총알배송과 유사한 의미  
    "신속 배송",      # 총알배송과 유사한 의미
    "총알배송",       # 정확한 키워드
]

print("🔍 키워드 검색 테스트:")
for kw in test_keywords:
    results = keyword_search(all_documents, kw)
    print(f"  '{kw}': {len(results)}개 문서 발견")

🔍 키워드 검색 테스트:
  '빠른 배송': 0개 문서 발견
  '배송 빨리': 0개 문서 발견
  '신속 배송': 0개 문서 발견
  '총알배송': 8개 문서 발견


### Test: 키워드 검색의 한계

위 결과에서 볼 수 있듯이:
- "총알배송" → 문서 발견 ✅
- "빠른 배송", "배송 빨리", "신속 배송" → 문서 못 찾음 ❌

**문제점**: 사용자가 정확한 키워드를 모르면 검색이 불가능합니다.

**정상 동작 기준:**
- 정확한 키워드("총알배송")로만 문서가 검색됨
- 유사한 의미의 다른 표현으로는 검색되지 않음

**예상 출력 예시:**
```
🔍 키워드 검색 테스트:
  '빠른 배송': 0개 문서 발견
  '배송 빨리': 0개 문서 발견
  '신속 배송': 0개 문서 발견
  '총알배송': 8개 문서 발견
```

**핵심 포인트:** 키워드 검색은 정확히 일치하는 단어만 찾습니다. 의미가 비슷한 표현도 찾으려면 어떻게 해야 할까요?

---
# Step 5: 의미 기반 검색 + DB화 기초

### Concept Check: 임베딩과 Vector Store

**임베딩(Embedding)**이란?
- 텍스트를 **숫자 벡터**로 변환하는 기술
- 의미가 비슷한 텍스트는 **비슷한 벡터**를 가짐
- 예: "총알배송" ≈ "빠른 배송" ≈ "신속 배송" (벡터 유사도 높음)

**Vector Store란?**
- 임베딩 벡터를 **저장하고 검색**하는 데이터베이스
- 쿼리 벡터와 **유사도가 높은 문서**를 빠르게 검색
- 대표 예: ChromaDB, Pinecone, FAISS

```
[의미 기반 검색 흐름]
질문: "빠른 배송 알려줘"
    ↓ 임베딩
질문 벡터: [0.2, -0.5, 0.8, ...]
    ↓ 유사도 검색
가장 유사한 문서: "총알배송_서비스 혜택.pdf" (유사도 0.92)
```

### 코사인 유사도(Cosine Similarity)의 수학적 배경

Vector Store에서 "유사한 문서"를 찾을 때 가장 많이 사용하는 방법이 **코사인 유사도**입니다.

**코사인 유사도 공식:**
$$\text{cosine\_similarity}(A, B) = \frac{A \cdot B}{||A|| \times ||B||} = \frac{\sum_{i=1}^{n} A_i B_i}{\sqrt{\sum_{i=1}^{n} A_i^2} \times \sqrt{\sum_{i=1}^{n} B_i^2}}$$

**직관적 이해:**
- 두 벡터 사이의 **각도**를 측정 (크기는 무시)
- 값의 범위: -1 ~ 1 (임베딩 벡터의 각 차원 값은 양수·음수 모두 가능하며, 코사인 유사도는 벡터 간 **방향 유사성**을 측정)
- **1에 가까울수록** 의미가 유사, **0에 가까울수록** 관련 없음

**왜 코사인 유사도인가?**
| 유사도 방식 | 특징 | 적합한 상황 |
|------------|------|-------------|
| 유클리드 거리 | 벡터 간 직선 거리 | 크기도 중요한 경우 |
| **코사인 유사도** | 벡터 간 각도 (크기 무시) | 텍스트 유사도 측정 |
| 내적(Dot Product) | 빠른 계산 | 정규화된 벡터 |

텍스트 임베딩에서 코사인 유사도가 선호되는 이유:
- 긴 문서와 짧은 문서를 **공정하게 비교** 가능
- 의미적 **방향성**만 비교하므로 직관적

### TODO 4: 문서를 Vector DB에 저장하기

- **요구사항**: 문서를 임베딩하여 ChromaDB에 저장
- **입력**: 로드된 전체 문서
- **출력**: Vector Store 객체
- **힌트**: `Chroma.from_documents()`, `OpenAIEmbeddings` 사용

**예상 결과:**
- `vectorstore`가 `Chroma` 객체여야 함
- `vectorstore._collection.count()`가 문서 수(페이지 수)와 동일해야 함
- 임베딩 생성에 시간이 소요될 수 있음 (문서 수에 따라 10~30초)

**정상 동작 확인 방법:**
- "✅ Vector Store 생성 완료"와 "📊 저장된 문서 수: N개"가 출력되면 성공
- N이 로드된 문서 수와 동일하면 정상

In [ ]:
# TODO: 문서를 Vector DB에 저장하기
from langchain_community.vectorstores import Chroma
import tiktoken

# 임베딩 모델의 토큰 제한 처리 (text-embedding-3-small: 최대 8,191 토큰)
# PDF 전체 페이지는 토큰 제한을 초과할 수 있으므로 사전에 잘라줌
MAX_TOKENS = 8000  # 안전 여유분 확보
enc = tiktoken.encoding_for_model("text-embedding-3-small")

truncated_count = 0
for doc in all_documents:
    tokens = enc.encode(doc.page_content)
    if len(tokens) > MAX_TOKENS:
        doc.page_content = enc.decode(tokens[:MAX_TOKENS])
        truncated_count += 1

if truncated_count > 0:
    print(f"⚠️ {truncated_count}개 문서가 토큰 제한 초과로 잘림 (최대 {MAX_TOKENS} 토큰)")

# Vector Store 생성 (문서 단위 - 페이지별)
vectorstore = Chroma.from_documents(
    documents=all_documents,
    embedding=embeddings,
    collection_name="yes24_docs_page"
)

print(f"✅ Vector Store 생성 완료")
print(f"📊 저장된 문서 수: {vectorstore._collection.count()}개")

✅ Vector Store 생성 완료
📊 저장된 문서 수: 8개


### TODO 5: 의미 기반 검색 수행하기

- **요구사항**: 사용자 질문으로 유사한 문서 검색
- **입력**: 검색 쿼리 (예: "빠른 배송", "포인트 적립")
- **출력**: 유사도가 높은 문서 목록
- **확인**: 키워드 검색과 비교하여 개선 확인
- **힌트**: `vectorstore.as_retriever()` 사용

**예상 결과:**
- `retriever`가 `VectorStoreRetriever` 객체여야 함
- "빠른 배송", "배송 빨리", "신속 배송" 등 유사 표현으로도 "총알배송" 관련 문서가 검색되어야 함
- 검색 결과에 "총알배송_서비스 혜택" 파일이 포함되어야 함

**정상 동작 확인 방법:**
- 모든 테스트 쿼리에서 "총알배송" 관련 파일이 검색되면 성공
- 키워드 검색(Step 4)에서 못 찾던 "빠른 배송"으로도 결과가 나오면 의미 기반 검색 동작 확인

In [ ]:
# TODO: 의미 기반 검색 수행하기

# Retriever 생성
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# 의미 기반 검색 테스트
test_queries = [
    "빠른 배송",      # 총알배송과 유사한 의미
    "배송 빨리",      # 총알배송과 유사한 의미  
    "신속 배송",      # 총알배송과 유사한 의미
    "총알배송",       # 정확한 키워드
]

print("🔍 의미 기반 검색 테스트:")
for query in test_queries:
    results = retriever.invoke(query)
    source_files = set([doc.metadata.get('source_file', 'Unknown') for doc in results])
    print(f"  '{query}': {source_files}")

🔍 의미 기반 검색 테스트:
  '빠른 배송': {'배송지연 보상제도_서비스 혜택 - 예스24.pdf', '매장 픽업 서비스_서비스 혜택 - 예스24.pdf', '총알배송_서비스 혜택 - 예스24.pdf'}
  '배송 빨리': {'배송지연 보상제도_서비스 혜택 - 예스24.pdf', '매장 픽업 서비스_서비스 혜택 - 예스24.pdf', '총알배송_서비스 혜택 - 예스24.pdf'}
  '신속 배송': {'배송지연 보상제도_서비스 혜택 - 예스24.pdf', '매장 픽업 서비스_서비스 혜택 - 예스24.pdf', '총알배송_서비스 혜택 - 예스24.pdf'}
  '총알배송': {'배송지연 보상제도_서비스 혜택 - 예스24.pdf', '매장 픽업 서비스_서비스 혜택 - 예스24.pdf', '총알배송_서비스 혜택 - 예스24.pdf'}


In [ ]:
# 검색 결과 상세 확인
if retriever:
    query = "배송이 늦으면 어떻게 되나요?"
    results = retriever.invoke(query)

    print(f"🔍 질문: {query}")
    print(f"\n📄 검색 결과 ({len(results)}개):")
    for i, doc in enumerate(results, 1):
        print(f"\n--- 결과 {i} ---")
        print(f"출처: {doc.metadata.get('source_file', 'Unknown')}")
        print(f"내용: {doc.page_content[:200]}...")

🔍 질문: 배송이 늦으면 어떻게 되나요?

📄 검색 결과 (3개):

--- 결과 1 ---
출처: 배송지연 보상제도_서비스 혜택 - 예스24.pdf
내용: 싸다
빠르다
믿을 수 있다
예스24에서 주문하신 아침/당일/하루 배송 상품이 주문 완료 시점에 안내된 도착 예상 일보다 지연된 경우 주문 건당 YES포인트 2,000원을 보상해 드리는 제도입니다. (2014년 11월 21일 주문부
터 시행)
보상대상
아침/당일/하루 배송 대상 상품 (예스24 직배송)
보상기준
주문한 아침/당일/하루 배송 상품이 도착예상일(...

--- 결과 2 ---
출처: 총알배송_서비스 혜택 - 예스24.pdf
내용: 싸다
빠르다
믿을 수 있다
전국 방방곡곡으로 총알배송 서비스는 계속 확대됩니다.
당일배송 표시상품을 주문하시면 당일에 받으실 수 있습니다.
주문일
원주
춘천
서울
수도권
(평택제외)
인천
대전, 세종
청주, 천안
아산, 평택
광주
울산, 창원
포항
부산, 김해
대구
경산
구미
도착
예상일
월~금
0~11시
0~13시
0~13시
0~12시
0~12시
0~12...

--- 결과 3 ---
출처: 도서 품절 보상제도 - 예스24.pdf
내용: 싸다
빠르다
믿을 수 있다
예스24에서 배송하는 도서(잡지, 중고도서, 직수입 도서 제외)가 일시품절, 품절, 절판, 판매금지, 미출간 상태로 변경된 경우, 주문취소 후 품절 상품별로 상품 금액에 따라 YES포인트 1,000원,
500원을 보상해 드리는 제도입니다. (2014년 11월 21일 주문부터 시행)
보상대상
예스24 직배송 도서 (잡지, 중고 , ...


### Test: 의미 기반 검색의 한계

의미 기반 검색은 키워드 검색보다 훨씬 유연하지만, 아직 문제가 있습니다:

- 현재 문서 단위(페이지별)로 저장
- 하나의 페이지에 여러 주제가 섞여 있을 수 있음
- 검색 결과가 **너무 크거나** 원하는 정보가 **묻혀있을 수 있음**

**정상 동작 기준:**
- 모든 유사 표현("빠른 배송", "신속 배송" 등)으로 "총알배송" 관련 문서가 검색됨
- 키워드 검색과 달리 의미적으로 유사한 쿼리도 동작함

**예상 출력 예시:**
```
🔍 의미 기반 검색 테스트:
  '빠른 배송': {'매장 픽업 서비스_서비스 혜택 - 예스24.pdf', '배송지연 보상제도_서비스 혜택 - 예스24.pdf', '총알배송_서비스 혜택 - 예스24.pdf'}
  '배송 빨리': {'매장 픽업 서비스_서비스 혜택 - 예스24.pdf', '배송지연 보상제도_서비스 혜택 - 예스24.pdf', '총알배송_서비스 혜택 - 예스24.pdf'}
  '신속 배송': {'매장 픽업 서비스_서비스 혜택 - 예스24.pdf', '배송지연 보상제도_서비스 혜택 - 예스24.pdf', '총알배송_서비스 혜택 - 예스24.pdf'}
  '총알배송': {'매장 픽업 서비스_서비스 혜택 - 예스24.pdf', '배송지연 보상제도_서비스 혜택 - 예스24.pdf', '총알배송_서비스 혜택 - 예스24.pdf'}
```

**핵심 포인트:** 의미 기반 검색은 유연하지만, 문서 단위가 크면 정확도가 떨어집니다. 더 작은 단위로 나누면 어떨까요?

---
# Step 6: 청킹 전략

### Concept Check: 청킹(Chunking)이란?

**청킹**은 긴 문서를 적절한 크기의 조각(chunk)으로 나누는 기법입니다.

**왜 청킹이 필요한가?**
1. **토큰 제한**: LLM에는 입력 토큰 제한이 있음
2. **검색 정확도**: 작은 단위일수록 관련성 높은 정보만 검색
3. **비용 효율**: 불필요한 토큰 사용 감소

**주요 파라미터:**
- `chunk_size`: 청크의 최대 크기 (문자 수)
- `chunk_overlap`: 청크 간 겹치는 부분 (문맥 유지)

```
[청킹 예시]
원본: "ABCDEFGHIJ" (10글자)
chunk_size=5, overlap=2
→ ["ABCDE", "DEFGH", "GHIJ"]
```

### 청킹 전략별 Trade-off 심화 분석

청킹 설정은 **검색 품질**, **비용**, **응답 정확도** 사이의 균형을 결정합니다.

**chunk_size Trade-off:**

| chunk_size | 장점 | 단점 | 적합한 상황 |
|------------|------|------|-------------|
| 작음 (100~200) | 세밀한 검색, 정확한 매칭 | 문맥 부족, 청크 수 증가 | FAQ, 정의/용어 검색 |
| 중간 (300~500) | 균형 잡힌 검색 | 최적점 찾기 어려움 | 일반 문서, 정책 안내 |
| 큼 (1000+) | 풍부한 문맥 | 노이즈 포함, 검색 정확도 하락 | 긴 설명문, 연속적 내용 |

**chunk_overlap Trade-off:**

| overlap 비율 | 장점 | 단점 |
|--------------|------|------|
| 0% | 중복 없음, 저장 효율적 | **문맥 단절** 위험, 문장 중간 절단 |
| 10~20% | 문맥 연결 유지, 적절한 중복 | 표준적인 선택 |
| 30%+ | 강한 문맥 유지 | **과도한 중복**, 저장/비용 증가 |

**실전 선택 기준:**

```
문서 특성 판단:
├── FAQ 형태 (짧은 Q&A)
│   └── chunk_size: 100~200, overlap: 0~10%
├── 정책 문서 (조건/단서 많음)
│   └── chunk_size: 300~400, overlap: 15~20%
├── 기술 문서 (연속적 설명)
│   └── chunk_size: 500~800, overlap: 10~15%
└── 법률/계약 문서 (정확성 중요)
    └── chunk_size: 200~300, overlap: 20~25%
```

**Yes24 서비스 문서의 경우:**
- 배송 정책, 포인트 제도 등 **조건부 설명**이 많음
- "~의 경우", "단," 등 **조건절**이 자주 등장
- **권장 설정**: chunk_size=300, overlap=50 (약 17%)

### TODO 6: 청킹 전략 비교하기

- **요구사항**: 다양한 chunk_size와 overlap으로 청킹하고 결과 비교
- **입력**: 문서, 청킹 설정 (chunk_size, overlap)
- **출력**: 청킹된 문서 조각들
- **비교**: 설정별 청크 수, 검색 품질 비교
- **힌트**: `RecursiveCharacterTextSplitter` 사용

**예상 결과:**
- `text_splitter`가 `RecursiveCharacterTextSplitter` 객체여야 함
- 권장 설정(chunk_size=300, chunk_overlap=50)으로 설정
- 청킹 후 조각 수가 원본 문서 수보다 많아야 함

**정상 동작 확인 방법:**
- "📊 청킹 결과:"에서 "청킹 후 조각 수"가 "원본 문서 수"보다 크면 성공
- 평균 청크 크기가 chunk_size(300)에 가깝게 출력되면 정상

In [ ]:
# TODO: 청킹 전략 비교하기
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 다양한 청킹 설정 비교
chunking_configs = [
    {"chunk_size": 100, "chunk_overlap": 0},
    {"chunk_size": 200, "chunk_overlap": 20},
    {"chunk_size": 500, "chunk_overlap": 50},
    {"chunk_size": 1000, "chunk_overlap": 100},
]

# 하나의 문서로 테스트
test_doc = all_documents[0]
print(f"📄 원본 문서 길이: {len(test_doc.page_content)} 글자\n")

for config in chunking_configs:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=config["chunk_size"],
        chunk_overlap=config["chunk_overlap"],
        length_function=len,
    )
    chunks = splitter.split_documents([test_doc])
    print(f"chunk_size={config['chunk_size']}, overlap={config['chunk_overlap']}: {len(chunks)}개 청크")

📄 원본 문서 길이: 1224 글자

chunk_size=100, overlap=0: 16개 청크
chunk_size=200, overlap=20: 7개 청크
chunk_size=500, overlap=50: 3개 청크
chunk_size=1000, overlap=100: 2개 청크


In [ ]:
# 최적 청킹 설정 선택 및 전체 문서 청킹
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50,
    length_function=len,
    separators=["\n\n", "\n", ".", " ", ""]
)

# 전체 문서 청킹
chunked_documents = text_splitter.split_documents(all_documents)

print(f"📊 청킹 결과:")
print(f"  - 원본 문서 수: {len(all_documents)}개")
print(f"  - 청킹 후 조각 수: {len(chunked_documents)}개")
print(f"  - 평균 청크 크기: {sum(len(doc.page_content) for doc in chunked_documents) / len(chunked_documents):.0f}자")

📊 청킹 결과:
  - 원본 문서 수: 8개
  - 청킹 후 조각 수: 56개
  - 평균 청크 크기: 271자


In [ ]:
# 청킹된 문서로 새 Vector Store 생성
vectorstore_chunked = Chroma.from_documents(
    documents=chunked_documents,
    embedding=embeddings,
    collection_name="yes24_docs_chunked"
)

retriever_chunked = vectorstore_chunked.as_retriever(search_kwargs={"k": 3})

print(f"✅ 청킹된 문서로 Vector Store 재생성 완료")
print(f"📊 저장된 청크 수: {vectorstore_chunked._collection.count()}개")

✅ 청킹된 문서로 Vector Store 재생성 완료
📊 저장된 청크 수: 56개


In [ ]:
# 청킹 전후 검색 품질 비교
query = "배송이 늦으면 보상받을 수 있나요?"

print(f"🔍 질문: {query}\n")

# 페이지 단위 검색
results_page = retriever.invoke(query)
print("📄 [페이지 단위 검색]")
print(f"  - 첫 번째 결과 길이: {len(results_page[0].page_content)}자")
print(f"  - 내용: {results_page[0].page_content[:150]}...")

print()

# 청크 단위 검색
results_chunk = retriever_chunked.invoke(query)
print("📄 [청크 단위 검색]")
print(f"  - 첫 번째 결과 길이: {len(results_chunk[0].page_content)}자")
print(f"  - 내용: {results_chunk[0].page_content[:150]}...")

🔍 질문: 배송이 늦으면 보상받을 수 있나요?

📄 [페이지 단위 검색]
  - 첫 번째 결과 길이: 1478자
  - 내용: 싸다
빠르다
믿을 수 있다
예스24에서 주문하신 아침/당일/하루 배송 상품이 주문 완료 시점에 안내된 도착 예상 일보다 지연된 경우 주문 건당 YES포인트 2,000원을 보상해 드리는 제도입니다. (2014년 11월 21일 주문부
터 시행)
보상대상
아침/당일/하루 배...

📄 [청크 단위 검색]
  - 첫 번째 결과 길이: 294자
  - 내용: 싸다
빠르다
믿을 수 있다
예스24에서 주문하신 아침/당일/하루 배송 상품이 주문 완료 시점에 안내된 도착 예상 일보다 지연된 경우 주문 건당 YES포인트 2,000원을 보상해 드리는 제도입니다. (2014년 11월 21일 주문부
터 시행)
보상대상
아침/당일/하루 배...


### 청킹 전략 선택 기준

| chunk_size | 특징 | 적합한 상황 |
|------------|------|-------------|
| 작음 (100~200) | 세밀한 검색, 청크 수 많음 | FAQ, 짧은 문단 |
| 중간 (300~500) | 균형 잡힌 검색 | 일반 문서 |
| 큼 (1000+) | 넓은 문맥, 청크 수 적음 | 긴 설명, 연속적 내용 |

**overlap 설정:**
- 0: 문맥 단절 위험
- chunk_size의 10~20%: 적절한 문맥 유지
- 너무 큼: 중복 정보, 비용 증가

---
# Step 7: 전체 RAG 파이프라인 (LangGraph)

### Concept Check: LangGraph란?

**LangGraph**는 LangChain 팀에서 개발한 **상태 기반 워크플로우 프레임워크**입니다.

**왜 LangGraph인가?**
1. **명시적 상태 관리**: 각 노드가 상태를 읽고 쓰며, 데이터 흐름이 명확함
2. **모듈화된 노드**: 검색, 생성 등 각 단계를 독립적인 노드로 분리
3. **확장성**: 조건부 분기, 반복, 병렬 실행 등 복잡한 워크플로우 지원
4. **디버깅 용이**: 각 노드의 입출력을 추적하기 쉬움

**LangGraph의 핵심 구성요소:**
- **State**: 그래프 전체에서 공유되는 데이터 구조
- **Node**: 상태를 변환하는 함수 (검색, 생성 등)
- **Edge**: 노드 간 연결 (순차 실행, 조건부 분기)

```
[LangGraph RAG 파이프라인 구조]

        START
          │
          ▼
    ┌─────────────┐
    │   retrieve  │  ← 질문으로 관련 문서 검색
    │   (Node 1)  │
    └─────────────┘
          │
          ▼
    ┌─────────────┐
    │   generate  │  ← 검색된 문서로 답변 생성
    │   (Node 2)  │
    └─────────────┘
          │
          ▼
         END
```

**LCEL 체인 vs LangGraph:**
| 비교 항목 | LCEL 체인 | LangGraph |
|----------|----------|-----------|
| 상태 관리 | 암시적 | 명시적 (TypedDict) |
| 분기/반복 | 제한적 | 유연함 |
| 디버깅 | 어려움 | 용이함 |
| 확장성 | 단순 파이프라인 적합 | 복잡한 워크플로우 적합 |

### TODO 7: LangGraph StateGraph로 RAG 파이프라인 구현

- **요구사항**: LangGraph의 StateGraph를 사용하여 검색 + 답변 생성 RAG 파이프라인 구현
- **입력**: 사용자 질문
- **출력**: 검색된 문서 기반의 정확한 답변
- **구현 단계**:
  1. State 타입 정의 (question, context, answer)
  2. retrieve 노드 구현 (검색)
  3. generate 노드 구현 (답변 생성)
  4. StateGraph로 노드 연결
- **확인**: Step 2 결과와 비교하여 RAG 효과 체감
- **힌트**: `StateGraph`, `TypedDict`, `START`, `END` 사용

**예상 결과:**
- `workflow`가 `StateGraph` 객체여야 함
- `rag_graph`가 컴파일된 그래프 객체여야 함
- 그래프 실행 결과에 `question`, `context`, `answer` 키가 포함되어야 함

**정상 동작 확인 방법:**
- "✅ LangGraph RAG 파이프라인 구성 완료!"가 출력되면 성공
- 질문에 대해 문서 기반의 구체적인 답변이 출력되면 정상
- Step 2(LLM만)와 비교했을 때 더 정확하고 구체적인 답변이 나오면 RAG 효과 확인

In [ ]:
# TODO: LangGraph StateGraph로 RAG 파이프라인 구현
from typing import TypedDict, List
from langgraph.graph import StateGraph, START, END
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate

# 1. State 타입 정의
# - question: 사용자 질문
# - context: 검색된 문서 내용
# - answer: 생성된 답변
class RAGState(TypedDict):
    question: str
    context: str
    answer: str

# 2. retrieve 노드: 질문으로 관련 문서 검색
def retrieve(state: RAGState) -> RAGState:
    """Vector Store에서 관련 문서를 검색하는 노드"""
    question = state["question"]
    
    # 청킹된 Vector Store에서 검색
    docs = retriever_chunked.invoke(question)
    
    # 검색 결과를 문자열로 포맷팅
    context = "\n\n".join([doc.page_content for doc in docs])
    
    return {"context": context}

# 3. generate 노드: 검색된 문서로 답변 생성
def generate(state: RAGState) -> RAGState:
    """검색된 문서를 기반으로 답변을 생성하는 노드"""
    question = state["question"]
    context = state["context"]
    
    # RAG 프롬프트 정의
    rag_prompt = ChatPromptTemplate.from_messages([
        ("system", """당신은 AI 온라인 서점 'Yes24'의 고객 서비스 상담원입니다.
다음 검색된 자료를 참고하여 고객의 질문에 정확하고 친절하게 답변해주세요.

[검색된 자료]
{context}

답변 규칙:
1. 검색된 자료를 기반으로 답변하세요
2. 자료에 없는 내용은 추측하지 마세요
3. 존댓말을 사용하세요
4. 간결하고 명확하게 답변하세요"""),
        ("human", "{question}")
    ])
    
    # 프롬프트 생성 후 LLM 직접 호출
    messages = rag_prompt.format_messages(context=context, question=question)
    response = llm.invoke(messages)
    answer = response.content
    
    return {"answer": answer}

# 4. StateGraph 구성
# - 노드 추가: retrieve, generate
# - 엣지 연결: START → retrieve → generate → END
workflow = StateGraph(RAGState)

# 노드 추가
workflow.add_node("retrieve", retrieve)
workflow.add_node("generate", generate)

# 엣지 연결 (순차 실행)
workflow.add_edge(START, "retrieve")
workflow.add_edge("retrieve", "generate")
workflow.add_edge("generate", END)

# 그래프 컴파일
rag_graph = workflow.compile()

print("✅ LangGraph RAG 파이프라인 구성 완료!")

✅ LangGraph RAG 파이프라인 구성 완료!


In [ ]:
# LangGraph RAG 파이프라인 테스트
question = "Yes24에서 총알배송이 뭔가요? 어떤 조건에서 가능한가요?"

print("📝 질문:", question)
print("\n🤖 LangGraph RAG 응답:")

# 그래프 실행
result = rag_graph.invoke({"question": question})
print(result["answer"])

📝 질문: Yes24에서 총알배송이 뭔가요? 어떤 조건에서 가능한가요?

🤖 LangGraph RAG 응답:
안녕하세요! Yes24 **총알배송**은 고객님께서 주문하신 책을 **매우 빠르게 받아볼 수 있는 배송 서비스**입니다.

### ✅ 총알배송이란?
- **주문 즉시 출고** 후, 지역별로 **익일 또는 당일** 수령이 가능합니다.
- 일반 배송보다 **훨씬 빠른 속도**로 배송되며, 물류센터에서 바로 처리되어 시간을 단축합니다.

---

### ✅ 총알배송 이용 조건
1. **대상 상품**  
   - 총알배송 대상 상품만 가능합니다 (예: 일부 인기 도서, 베스트셀러 등)
2. **구매 금액 제한 없음**  
   - 구매 금액에 관계없이 배송비 **무료**로 이용 가능
3. **배송 지역**  
   - 서울, 수도권, 충청권, 호남권(광주, 군산, 익산, 전주), 영남권(부산, 대구, 울산, 창원, 김해, 양산)  
   - 그 외 지역은 일부 조정 가능성이 있음
4. **주문 시간**  
   - **서울, 수도권, 인천**: 토요일 밤 21시까지 주문 시 월요일 또는 화요일 수령  
   - 그 외 지역: 토요일 밤 18시까지 주문 시 월요일 또는 화요일 수령

---

### ✅ 매장 픽업 서비스와 연계 가능
- **총알배송 대상 상품**이라면, 주문 시 **매장 픽업**도 함께 선택할 수 있습니다.
- 매장 픽업 시 **배송비 무료**이며, PC/모바일에서 구매한 상품을 가까운 예스24 매장에서 직접 받아볼 수 있습니다.

---

### ✅ 총알배송 혜택
- **아침배송 상품**: 오늘 밤 10시 전 주문 시 **내일 아침 7시 전** 수령 가능

---

추가로 궁금한 점이 있으시면 언제든지 문의 주세요! 😊


In [ ]:
# 다양한 질문 테스트
test_questions = [
    "배송이 늦으면 어떻게 보상받을 수 있나요?",
    "YES포인트는 어떻게 적립되나요?",
    "신규 회원 혜택은 무엇인가요?",
    "도서가 품절되면 어떻게 되나요?"
]

for q in test_questions:
    print(f"\n{'='*50}")
    print(f"📝 질문: {q}")
    print(f"\n🤖 LangGraph RAG 응답:")
    result = rag_graph.invoke({"question": q})
    print(result["answer"])


📝 질문: 배송이 늦으면 어떻게 보상받을 수 있나요?

🤖 LangGraph RAG 응답:
배송이 늦을 경우, 다음과 같은 절차로 보상받으실 수 있습니다.

1. **보상 대상 조건**  
   - 예스24 직배송 상품 중 **아침/당일/하루 배송**으로 주문하신 경우  
   - 주문 완료 시 안내된 도착 예상일보다 **이상 지연**된 경우  
   - 주문 상태가 **출고 완료**이며, **출고 완료일로부터 14일 이내**에 신청해야 합니다.

2. **보상 금액**  
   - 주문 건당 **YES포인트 2,000원**이 지급됩니다.

3. **신청 방법**  
   - 예스24 홈페이지 또는 모바일 앱에서 **배송지연 보상 신청**을 진행하시면 됩니다.  
   - 신청 후 **익일에 자동으로 YES포인트가 지급**됩니다. (단, 택배사 배송 결과 확인 지연 시 지급이 다소 늦어질 수 있습니다.)

4. **제외 사항**  
   - 주문 취소/반품 처리한 주문  
   - 비회원 주문  
   - 제휴사 주문 또는 고객 부재 등 고객 사유로 인한 지연  
   - 명절 성수기(설날, 추석 등)로 인한 지연  
   - 재고조사, 시스템 점검 등으로 보상 서비스가 불가능한 경우  

자세한 사항은 예스24 홈페이지에서 확인하실 수 있습니다.

📝 질문: YES포인트는 어떻게 적립되나요?

🤖 LangGraph RAG 응답:
YES포인트는 다음과 같은 방법으로 적립됩니다:

1. **도서사이트 내 결제 시 적립**
   - **예스24 간편결제 사용 시**: YES포인트 3% 적립
   - **일반결제 사용 시**: YES포인트 2% 적립
   - **일반 가맹점 결제 시**: YES포인트 1% 적립

2. **추가 적립 혜택**
   - 도서, eBook, CD/LP, DVD/Blu-ray, 문구/GIFT 상품을 **5만원 이상 주문 시 추가 적립** 혜택 제공

3. **특징**
   - YES포인트는 **유효기간 없이 영원히** 사용 가능합니다.


### Test: Step 2 vs Step 7 비교 (RAG 효과 체감)

같은 질문으로 Step 2(LLM만)와 Step 7(RAG)의 결과를 비교합니다.

**정상 동작 기준:**
- Step 7(RAG)의 응답이 Step 2(LLM만)보다 구체적이고 정확해야 함
- 실제 문서 내용(배송 지연 보상 조건, 금액 등)이 포함되어야 함
- LLM만 사용할 때 나타나던 불확실한 표현이 줄어들어야 함

**예상 출력 예시:**
```
📊 Step 2 vs Step 7 비교
==================================================

📝 질문: Yes24 배송지연 보상제도는 어떻게 되나요?

[Step 2: LLM만 사용]
Yes24의 배송지연 보상제도에 대해서는 정확한 정보가 없습니다.
일반적으로 온라인 서점에서는... (추측성 답변)

[Step 7: LangGraph RAG 사용]
Yes24 배송지연 보상제도는 다음과 같습니다:
- 총알배송 지연 시 YES포인트 500원 적립
- 일반배송 지연 시 YES포인트 300원 적립
- 보상 신청은 배송 완료 후 7일 이내
(문서에 있는 실제 정책 기반 - 정확한 내용은 문서에 따라 다름)
```

In [ ]:
# Step 2 vs Step 7 비교
comparison_question = "Yes24 배송지연 보상제도는 어떻게 되나요?"

print("="*60)
print("📊 Step 2 vs Step 7 비교")
print("="*60)
print(f"\n📝 질문: {comparison_question}")

# Step 2: LLM만
print("\n[Step 2: LLM만 사용]")
response_llm_only = llm.invoke([HumanMessage(content=comparison_question)])
print(response_llm_only.content)

# Step 7: LangGraph RAG
print("\n[Step 7: LangGraph RAG 사용]")
result = rag_graph.invoke({"question": comparison_question})
print(result["answer"])

📊 Step 2 vs Step 7 비교

📝 질문: Yes24 배송지연 보상제도는 어떻게 되나요?

[Step 2: LLM만 사용]
## Yes24(인터파크) 배송 지연 보상 제도 안내  

> **※ 현재(2026년 3월) Yes24는 인터파크의 “인터파크 쇼핑” 플랫폼을 통해 운영하고 있습니다. 따라서 보상 정책은 인터파크 쇼핑(인터파크) 배송 정책에 따릅니다.**  
> 아래 내용은 2025‑07까지 확인된 정책과 2026년 3월 기준 실무 운영 방식을 종합한 것이며, 구체적인 상황에 따라 차이가 있을 수 있으니 구매 전 ‘배송 정책’ 페이지를 반드시 확인하시기 바랍니다.

---

### 1. 기본 보상 원칙  

| 구분 | 보상 내용 | 적용 조건 |
|------|-----------|-----------|
| **배송 지연** | • 주문 후 **2일** 이상 도착 시(다음 영업일 기준) **배송비 전액 전액**을 포인트로 환급하거나 쿠폰 형태로 지급 <br>• 5일 이상 지연 시 **구매 금액의 5 %**를 포인트로 추가 보상 <br>• 10일 이상 지연 시 **구매 금액의 10 %**를 포인트로 추가 보상 | • 주문·배송 상태가 “배송 준비 중 → 출고 → 배송 중”으로 정상적으로 진행되었으나, **예상 도착일**(주문 상세 페이지·문자에 표기)보다 **2일 이상 초과**했을 경우 <br>• 지연 사유가 Yes24·인터파크의 책임(배송 오류·재고 부족·택배사 실수 등)인 경우 |
| **배송 지연 사유** | • 배송 지역(예: 제주·도서산간)·날씨·택배사 파업·통관 문제 등 **외부 요인**은 보상 대상에서 제외될 수 있음 (단, 외부 요인이라도 2일 이상 지연되면 정책에 따라 보상 가능) | • “예상 도착일”이 **인터파크가 제공한 배송 예정일**이어야 함 |
| **보상 형태** | • **인터파크 쇼핑 포인트** (구매 시 바로 사용 가능) <br>• **인터파크 쿠폰**(다음 주문 시 사용) <br>• 경우에 따라 **전액 현금 

### RAG 효과 정리

| 비교 항목 | LLM만 (Step 2) | LangGraph RAG (Step 7) |
|----------|----------------|-------------------------|
| 정확도 | 낮음 (추측/환각) | 높음 (자료 기반) |
| 신뢰도 | 낮음 | 높음 |
| 최신 정보 | 학습 시점까지만 | 업데이트 가능 |
| 도메인 지식 | 제한적 | 문서에 따라 확장 |
| 출처 추적 | 불가능 | 가능 |
| 확장성 | 제한적 | 노드 추가로 유연한 확장 |

**LangGraph의 장점:**
- 각 노드(retrieve, generate)의 역할이 명확히 분리됨
- 상태(State)를 통해 데이터 흐름이 투명함
- 향후 조건부 분기, 반복 등 복잡한 로직 추가 용이

**핵심 포인트:** LangGraph RAG를 통해 정확한 답변 생성이 가능해졌고, 워크플로우 확장도 용이합니다.

---
## 트러블슈팅 가이드

실습/과제 진행 중 자주 발생하는 오류와 해결 방법입니다.

### API 관련

| 증상 | 원인 | 해결 방법 |
|------|------|----------|
| `AuthenticationError` | API Key 미설정 또는 잘못된 키 | `.env` 파일에 `OPENAI_API_KEY` 확인 |
| `RateLimitError` | API 호출 한도 초과 | 잠시 대기 후 재실행 (1~2분) |
| `InvalidRequestError` | 토큰 제한 초과 | 입력 텍스트 길이 줄이기 또는 청킹 크기 조정 |

### 패키지 관련

| 증상 | 원인 | 해결 방법 |
|------|------|----------|
| `ModuleNotFoundError` | 패키지 미설치 | Step 1의 `%pip install` 셀 재실행 |
| `ImportError: cannot import name` | 버전 불일치 | `%pip install --upgrade <패키지>` |
| ChromaDB `sqlite3` 오류 | SQLite 버전 낮음 | `%pip install pysqlite3-binary` 후 커널 재시작 |

### 일반

| 증상 | 원인 | 해결 방법 |
|------|------|----------|
| `NameError: name 'xxx' is not defined` | 이전 셀 미실행 | Step 1부터 순서대로 재실행 |
| 출력이 비어있음 | 환경 변수 로드 실패 | `.env` 파일 경로 확인 후 커널 재시작 |


---
# 실습 마무리

## 학습 내용 정리

이번 실습에서 배운 내용:

1. **Step 2**: LLM의 한계 체감 (환각, 도메인 지식 부재)
2. **Step 3**: 컨텍스트 추가로 해결 가능함 확인
3. **Step 4**: 키워드 검색의 한계 체감
4. **Step 5**: 의미 기반 검색(임베딩, Vector Store)으로 개선
5. **Step 6**: 청킹 전략으로 검색 품질 향상
6. **Step 7**: LangGraph StateGraph로 RAG 파이프라인 구현 및 효과 확인

## 학생용 자가 체크리스트

- [ ] LLM의 한계(환각, 도메인 지식 부재)를 설명할 수 있다
- [ ] 키워드 검색과 의미 기반 검색의 차이를 설명할 수 있다
- [ ] 임베딩과 Vector Store의 역할을 설명할 수 있다
- [ ] 청킹의 필요성과 chunk_size/overlap 선택 기준을 설명할 수 있다
- [ ] LangGraph의 State, Node, Edge 개념을 설명할 수 있다
- [ ] LangGraph StateGraph로 RAG 파이프라인을 구현할 수 있다
- [ ] 모든 TODO 코드가 에러 없이 실행된다

---

### **Content License Agreement**

<font color='red'><b>**WARNING**</b></font> : 본 자료는 삼성청년SW·AI아카데미의 컨텐츠 자산으로, 보안서약서에 의거하여 어떠한 사유로도 임의로 복사, 촬영, 녹음, 복제, 보관, 전송하거나 허가 받지 않은 저장매체를 이용한 보관, 제3자에게 누설, 공개 또는 사용하는 등의 무단 사용 및 불법 배포 시 법적 조치를 받을 수 있습니다.